<a href="https://colab.research.google.com/github/sevaradevoloper/machine-learning-class-work/blob/main/61dars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gzip, urllib.request, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"

def load(name, skip):
    urllib.request.urlretrieve(URL + name, name)
    return torch.tensor(np.frombuffer(gzip.open(name).read(), np.uint8, offset=skip).copy())

X = load("train-images-idx3-ubyte.gz", 16).float().div(255).view(-1, 1, 28, 28)
y = load("train-labels-idx1-ubyte.gz", 8).long()
Xte = load("t10k-images-idx3-ubyte.gz", 16).float().div(255).view(-1, 1, 28, 28)
yte = load("t10k-labels-idx1-ubyte.gz", 8).long()
Xtr, ytr, Xva, yva = X[:20000], y[:20000], X[20000:25000], y[20000:25000]
print(Xtr.shape, Xva.shape, Xte.shape)

torch.Size([20000, 1, 28, 28]) torch.Size([5000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


In [4]:
def block(i,o):
  return [nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(),nn.MaxPool2d(2)]
def cnn():
  return nn.Sequential(*block(1,16),*block(16,32),nn.Flatten(),nn.Dropout(0.3),nn.Linear(32*7*7,10))

print(sum(q.numel() for q in cnn().parameters()))

20586


In [5]:
def train_epoch(model, opt, sched=None, bs=128):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), bs):
        idx = perm[i:i + bs]
        xb = Xtr[idx]
        loss = F.cross_entropy(model(xb), ytr[idx])
        opt.zero_grad();
        loss.backward();
        opt.step()
        if sched:
            sched.step()

def val_acc(model):
    model.eval()
    with torch.no_grad():
        return (model(Xva).argmax(1) == yva).float().mean().item()

In [8]:
def fit(model , opt, epochs=10, *kw):
  accs = []
  for e in range(epochs):
    train_epoch(model, opt,**kw)
    accs.append(val_acc(model))
  return accs

def run(make_opt , make_sched=None, **kw):
  torch.manual_seed(0)
  model = cnn()
  opt = make_opt(model)
  sched = make_sched(opt) if make_sched else None
  return fit(model, opt, **kw, sched=sched)